In [1]:
import sys
import os
import warnings
from datetime import datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pytz
import json
from zoneinfo import ZoneInfo
from pandas import Timestamp
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
from scipy.stats import norm

current_dir = Path.cwd()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))
from lib import *

MODEL_PATH=parent_dir / 'models' 
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)

In [2]:
example_ticker = "KXBTC15M-26MAY040515-15"
lookback_minutes = 10000
series, event_dt = parse_kalshi_15m_event_ticker(example_ticker)
dt_only = get_ticker_datetime(example_ticker)
# crypto_at = datetime(2026,5,5,0,0,tzinfo=ZoneInfo('America/Chicago'))
crypto_at = datetime.now(tz=ZoneInfo('America/Chicago'))
df_api = get_market_data_from_api(series, crypto_at, lookback_minutes)
df_api = df_api.set_index('datetime')
df_api.head()

API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting market candlesticks: 404 Client Error: Not Found for url: https://api.elections.kalshi.com/trade-api/v2/series/KXBTC15M/markets/KXBTC15M-26MAY210215-15/candlesticks?series_ticker=KXBTC15M&ticker=KXBTC15M-26MAY210215-15&start_ts=1779342900&end_ts=1779344100&period_interval=1&include_latest_before_start=True&limit=1000
API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting market candlesticks: 404 Client Error: Not Found for url: https://api.elections.kalshi.com/trade-api/v2/series/KXBTC15M/markets/KXBTC15M-26MAY210230-30/candlesticks?series_ticker=KXBTC15M&ticker=KXBTC15M-26MAY210230-30&start_ts=1779343800&end_ts=1779345000&period_interval=1&include_latest_before_start=True&limit=1000
API Error Response: {'error': {'code': 'not_found', 'message': 'not found', 'service': 'query-exchange'}}
Error getting ma

,ticker,floor_strike,volume_fp,open_interest_fp,yes_ask_open_dollar,yes_ask_high_dollar,yes_ask_low_dollar,yes_ask_close_dollar,yes_bid_open_dollar,yes_bid_high_dollar,yes_bid_low_dollar,yes_bid_close_dollar,no_ask_open_dollar,no_ask_high_dollar,no_ask_low_dollar,no_ask_close_dollar,no_bid_open_dollar,no_bid_high_dollar,no_bid_low_dollar,no_bid_close_dollar
datetime,,,,,,,,,,,,,,,,,,,,
2026-05-16 13:46:00-05:00,KXBTC15M-26MAY161500-00,78236.47,19917.54,17044.97,0.999,0.999,0.38,0.38,0.00,0.48,0.00,0.37,0.63,0.52,1.00,1.00,0.62,0.001,0.62,0.001
2026-05-16 13:47:00-05:00,KXBTC15M-26MAY161500-00,78236.47,23708.44,36504.48,0.380,0.420,0.38,0.41,0.37,0.40,0.37,0.40,0.60,0.60,0.63,0.63,0.59,0.580,0.62,0.620
2026-05-16 13:48:00-05:00,KXBTC15M-26MAY161500-00,78236.47,17160.48,49749.90,0.410,0.460,0.41,0.44,0.40,0.44,0.40,0.43,0.57,0.56,0.60,0.60,0.56,0.540,0.59,0.590
2026-05-16 13:49:00-05:00,KXBTC15M-26MAY161500-00,78236.47,22812.78,49237.02,0.440,0.590,0.43,0.57,0.43,0.58,0.41,0.56,0.44,0.42,0.59,0.57,0.43,0.410,0.57,0.560
2026-05-16 13:50:00-05:00,KXBTC15M-26MAY161500-00,78236.47,23922.68,52366.07,0.570,0.620,0.32,0.32,0.56,0.60,0.31,0.31,0.69,0.40,0.69,0.44,0.68,0.380,0.68,0.430


In [3]:
df_crypto = get_crypto_past_minutes(series, crypto_at, lookback_minutes)
df_crypto['datetime'] = pd.to_datetime(df_crypto['datetime'])
df_crypto['datetime'] = df_crypto['datetime'].dt.tz_convert('America/Chicago')
df_crypto['datetime'] = df_crypto['datetime'].dt.floor('min')
df_crypto = df_crypto.set_index('datetime')
filter_timestamp = df_crypto[df_crypto.index.minute.isin([0,15,30,45])].index[0]
df_crypto = df_crypto[df_crypto.index >= filter_timestamp]
df_crypto.head()

,open,high,low,close,tick_count
datetime,,,,,
2026-05-11 11:45:00-05:00,81623.60,81623.88,81548.21,81569.80,41
2026-05-11 11:46:00-05:00,81565.10,81654.75,81559.09,81627.50,52
2026-05-11 11:47:00-05:00,81621.83,81632.41,81617.68,81618.15,12
2026-05-11 11:48:00-05:00,81622.48,81622.48,81622.48,81622.48,0
2026-05-11 11:49:00-05:00,81683.92,81683.92,81683.92,81683.92,0


In [4]:
df_merged = df_crypto.join(df_api, how='left')
df_merged = df_merged.dropna()
df_merged.head()

,open,high,low,close,tick_count,ticker,floor_strike,volume_fp,open_interest_fp,yes_ask_open_dollar,yes_ask_high_dollar,yes_ask_low_dollar,yes_ask_close_dollar,yes_bid_open_dollar,yes_bid_high_dollar,yes_bid_low_dollar,yes_bid_close_dollar,no_ask_open_dollar,no_ask_high_dollar,no_ask_low_dollar,no_ask_close_dollar,no_bid_open_dollar,no_bid_high_dollar,no_bid_low_dollar,no_bid_close_dollar
datetime,,,,,,,,,,,,,,,,,,,,,,,,,
2026-05-16 13:46:00-05:00,78229.56,78229.56,78229.14,78229.14,0,KXBTC15M-26MAY161500-00,78236.47,19917.54,17044.97,0.999,0.999,0.38,0.38,0.00,0.48,0.00,0.37,0.63,0.52,1.00,1.00,0.62,0.001,0.62,0.001
2026-05-16 13:47:00-05:00,78220.09,78220.09,78220.09,78220.09,0,KXBTC15M-26MAY161500-00,78236.47,23708.44,36504.48,0.380,0.420,0.38,0.41,0.37,0.40,0.37,0.40,0.60,0.60,0.63,0.63,0.59,0.580,0.62,0.620
2026-05-16 13:48:00-05:00,78222.18,78227.90,78222.18,78226.89,12,KXBTC15M-26MAY161500-00,78236.47,17160.48,49749.90,0.410,0.460,0.41,0.44,0.40,0.44,0.40,0.43,0.57,0.56,0.60,0.60,0.56,0.540,0.59,0.590
2026-05-16 13:49:00-05:00,78227.48,78227.48,78227.30,78227.30,1,KXBTC15M-26MAY161500-00,78236.47,22812.78,49237.02,0.440,0.590,0.43,0.57,0.43,0.58,0.41,0.56,0.44,0.42,0.59,0.57,0.43,0.410,0.57,0.560
2026-05-16 13:50:00-05:00,78246.53,78249.21,78234.33,78234.84,19,KXBTC15M-26MAY161500-00,78236.47,23922.68,52366.07,0.570,0.620,0.32,0.32,0.56,0.60,0.31,0.31,0.69,0.40,0.69,0.44,0.68,0.380,0.68,0.430


In [5]:
df_calc = df_merged

In [6]:
def rsi_calc(series, periods=14):
    delta = series.diff()
    gain = (delta.where(delta>0,0)).rolling(periods).mean()
    loss = (delta.where(delta<0,0)).rolling(periods).mean()
    rs = gain / loss
    rsi = 100 - (100/(1+rs))
    return rsi

In [7]:
def macd_calc(series, fast=14, slow=26, signal=13):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

In [8]:
research_side = 'yes'
df_calc['rsi'] = rsi_calc(df_calc['close'], periods=5) 
df_calc['macd'], df_calc['signal_line'], _ = macd_calc(df_calc['close'], 4, 9, 2) 
df_calc[research_side + '_dist'] = df_calc['close'] - df_calc['floor_strike']
df_calc['log_return'] = np.log(df_calc['close'] / df_calc['close'].shift(1))
df_calc['m3_log_return'] = df_calc['log_return'].rolling(3).std()
df_calc['m5_log_return'] = df_calc['log_return'].rolling(5).std()
df_calc['ma3'] = df_calc['close'].rolling(3).mean()
df_calc['ma5'] = df_calc['close'].rolling(5).mean()
df_calc['ma3_vs_strike'] = (df_calc['ma3'] - df_calc['floor_strike'])/df_calc['floor_strike'] * 100
df_calc['ma5_vs_strike'] = (df_calc['ma5'] - df_calc['floor_strike'])/df_calc['floor_strike'] * 100
df_calc[research_side + '_dist_pct'] = df_calc[research_side + '_dist'] / df_calc['floor_strike'] * 100
df_calc['m1_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(1)
df_calc['m3_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(3)
df_calc['m5_' + research_side + '_dist_momentum'] = df_calc[research_side + '_dist'] - df_calc[research_side + '_dist'].shift(5)
df_calc['time_decay'] = np.where(df_calc.index.minute % 15 == 0, 0, 15 - df_calc.index.minute % 15)
df_calc['hour'] = df_calc.index.hour
df_calc['minute'] = df_calc.index.minute
df_calc[research_side + '_spread'] = df_calc[research_side + '_ask_close_dollar'] - df_calc[research_side + '_bid_close_dollar']
df_calc['volume_surge'] = df_calc['volume_fp'] / df_calc['volume_fp'].rolling(5).mean()
df_calc['ask_delta'] = df_calc[research_side + '_ask_low_dollar'].diff()
df_calc['bid_delta'] = df_calc[research_side + '_bid_high_dollar'].diff()
df_calc['ask_std'] = df_calc[research_side + '_ask_low_dollar'].rolling(3).std()
df_calc['bid_std'] = df_calc[research_side + '_bid_high_dollar'].rolling(3).std()
# df_calc['oi_change'] = df_calc['open_interest_fp'] - df_calc['open_interest_fp'].shift(1)

In [9]:
df_calc = df_calc.dropna()

In [10]:
df_calc.head()

,open,high,low,close,tick_count,ticker,floor_strike,volume_fp,open_interest_fp,yes_ask_open_dollar,yes_ask_high_dollar,yes_ask_low_dollar,yes_ask_close_dollar,yes_bid_open_dollar,yes_bid_high_dollar,yes_bid_low_dollar,yes_bid_close_dollar,no_ask_open_dollar,no_ask_high_dollar,no_ask_low_dollar,no_ask_close_dollar,no_bid_open_dollar,no_bid_high_dollar,no_bid_low_dollar,no_bid_close_dollar,rsi,macd,signal_line,yes_dist,log_return,m3_log_return,m5_log_return,ma3,ma5,ma3_vs_strike,ma5_vs_strike,yes_dist_pct,m1_yes_dist_momentum,m3_yes_dist_momentum,m5_yes_dist_momentum,time_decay,hour,minute,yes_spread,volume_surge,ask_delta,bid_delta,ask_std,bid_std
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-05-16 13:51:00-05:00,78210.49,78242.44,78210.21,78241.03,55,KXBTC15M-26MAY161500-00,78236.47,25382.20,59247.27,0.32,0.540,0.32,0.54,0.31,0.530,0.31,0.53,0.47,0.470,0.69,0.69,0.46,0.460,0.68,0.68,176.114382,3.112784,2.224786,4.56,0.000079,0.000048,0.000089,78234.390000,78230.030,-0.002659,-0.008231,0.005828,6.19,14.14,11.89,9,13,51,0.01,1.123240,0.00,-0.070,0.063509,0.036056
2026-05-16 13:52:00-05:00,78238.47,78255.38,78235.16,78254.43,60,KXBTC15M-26MAY161500-00,78236.47,25990.18,60521.71,0.54,0.650,0.46,0.65,0.53,0.640,0.45,0.64,0.36,0.360,0.55,0.47,0.35,0.350,0.54,0.46,100.000000,6.509130,5.081015,17.96,0.000171,0.000049,0.000059,78243.433333,78236.898,0.008900,0.000547,0.022956,13.40,27.13,34.34,8,13,52,0.01,1.127377,0.14,0.110,0.080829,0.055678
2026-05-16 13:53:00-05:00,78250.35,78250.35,78249.52,78249.52,0,KXBTC15M-26MAY161500-00,78236.47,45794.19,66959.97,0.65,0.920,0.65,0.86,0.64,0.919,0.64,0.85,0.15,0.081,0.36,0.36,0.14,0.080,0.35,0.35,121.696863,6.636645,6.118102,13.05,-0.000063,0.000118,0.000090,78248.326667,78241.424,0.015155,0.006332,0.016680,-4.91,14.68,22.63,7,13,53,0.01,1.591159,0.19,0.279,0.165630,0.200525
2026-05-16 13:54:00-05:00,78281.56,78307.83,78280.59,78287.22,22,KXBTC15M-26MAY161500-00,78236.47,51480.36,72507.14,0.86,0.947,0.73,0.81,0.85,0.941,0.72,0.80,0.20,0.059,0.28,0.15,0.19,0.053,0.27,0.14,108.194259,13.706921,11.177315,50.75,0.000482,0.000273,0.000202,78263.723333,78253.408,0.034835,0.021650,0.064867,37.70,46.19,59.92,6,13,54,0.01,1.491582,0.08,0.022,0.138684,0.167793
2026-05-16 13:55:00-05:00,78272.71,78274.83,78270.35,78270.35,4,KXBTC15M-26MAY161500-00,78236.47,46691.54,87338.23,0.81,0.860,0.21,0.22,0.80,0.840,0.20,0.21,0.79,0.160,0.80,0.20,0.78,0.140,0.79,0.19,161.334835,12.630100,12.145838,33.88,-0.000216,0.000366,0.000263,78269.030000,78260.510,0.041617,0.030727,0.043305,-16.87,15.92,35.51,5,13,55,0.01,1.195145,-0.52,-0.101,0.280000,0.053113


In [17]:
def agg_data_function(df, column, *data_cents):
    results = {cent: [] for cent in data_cents}
    triggered = set() 
    
    for i in range(len(df)):
        row = df.iloc[i]
        
        if row['time_decay'] == 0:
            continue
        
        current_ticker = row['ticker']
        
        for cent in data_cents:
            if (current_ticker, cent) in triggered:
                continue
                
            if float(row[column + '_ask_low_dollar']) < float(cent):
                # print(f"trigger: {column + '_ask_low_dollar'}: {row[column + '_ask_low_dollar']}")
                triggered.add((current_ticker, cent)) 
                
                high_price = 0
                low_price = 1
                
                for j in range(1, 16):
                    if i + j >= len(df):
                        break
                    next_row = df.iloc[i + j]
                    high_price = max(next_row[column + '_bid_high_dollar'], high_price)
                    low_price = min(next_row[column + '_ask_low_dollar'], low_price)
                    if next_row['ticker'] != current_ticker or next_row['time_decay'] == 0:
                        break
                
                tmp_dict = row.to_dict()
                tmp_dict['subsequent_high'] = high_price
                tmp_dict['subsequent_low'] = low_price
                tmp_dict['reached_30'] = 1 if high_price >= 0.30 else 0
                tmp_dict['reached_40'] = 1 if high_price >= 0.40 else 0
                tmp_dict['reached_50'] = 1 if high_price >= 0.50 else 0
                tmp_dict['reached_60'] = 1 if high_price >= 0.60 else 0
                tmp_dict['reached_70'] = 1 if high_price >= 0.7 else 0
                tmp_dict['reached_90'] = 1 if high_price >= 0.9 else 0
                tmp_dict['reached_99'] = 1 if high_price >= 0.99 else 0
                results[cent].append(tmp_dict)
                
    
    return results

In [18]:
res=agg_data_function(df_calc, research_side, *[0.05,0.1,0.15,0.2,0.25,0.3,0.35])

In [19]:
# win rate analysis
comb_05 = pd.DataFrame(res[0.05])
comb_10 = pd.DataFrame(res[0.10])
comb_15 = pd.DataFrame(res[0.15])
comb_20 = pd.DataFrame(res[0.2])
comb_25 = pd.DataFrame(res[0.25])
comb_30 = pd.DataFrame(res[0.3])
comb_35 = pd.DataFrame(res[0.35])

In [20]:
def get_protential_pnl(df, entry_price, *exit_price):
    for price in exit_price:
        col_name = f"reached_{int(price * 100)}"
        if col_name in df.columns:
            rate = (df[col_name] == 1).sum() / len(df)
            pnl = rate * (float(price) - float(entry_price))
            print(f"Entry: {entry_price}, Exit: {price}, win rate: {rate:.2%}, PNL: {pnl:.4f}")

In [21]:
def evaluate_feature(good_mean, good_std, bad_mean, bad_std):
    diff = good_mean - bad_mean
    avg_std = (good_std + bad_std) / 2
    
    if avg_std < 1e-10:
        return None
    
    ratio = abs(diff) / avg_std
    
    if ratio > 0.5:
        grade = 'strong'
    elif ratio > 0.3:
        grade = 'medium'
    elif ratio > 0.15:
        grade = 'weak'
    else:
        grade = 'useless'
    
    return ratio, grade

In [38]:
good_hours = [9,10,11,12,13,14,15,16,20]

get_protential_pnl(comb_05[comb_05['hour'].isin(good_hours)],0.05,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_10[comb_10['hour'].isin(good_hours)],0.10,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_15[comb_15['hour'].isin(good_hours)],0.15,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_20[comb_20['hour'].isin(good_hours)],0.20,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_25[comb_25['hour'].isin(good_hours)],0.25,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_30[comb_30['hour'].isin(good_hours)],0.30,*[0.4,0.5,0.6,0.7,0.9,0.99])
get_protential_pnl(comb_35[comb_35['hour'].isin(good_hours)],0.35,*[0.4,0.5,0.6,0.7,0.9,0.99])

Entry: 0.05, Exit: 0.4, win rate: 14.04%, PNL: 0.0491
Entry: 0.05, Exit: 0.5, win rate: 12.28%, PNL: 0.0553
Entry: 0.05, Exit: 0.6, win rate: 10.53%, PNL: 0.0579
Entry: 0.05, Exit: 0.7, win rate: 10.53%, PNL: 0.0684
Entry: 0.05, Exit: 0.9, win rate: 8.77%, PNL: 0.0746
Entry: 0.05, Exit: 0.99, win rate: 8.77%, PNL: 0.0825
Entry: 0.1, Exit: 0.4, win rate: 24.24%, PNL: 0.0727
Entry: 0.1, Exit: 0.5, win rate: 22.73%, PNL: 0.0909
Entry: 0.1, Exit: 0.6, win rate: 21.21%, PNL: 0.1061
Entry: 0.1, Exit: 0.7, win rate: 19.70%, PNL: 0.1182
Entry: 0.1, Exit: 0.9, win rate: 18.18%, PNL: 0.1455
Entry: 0.1, Exit: 0.99, win rate: 16.67%, PNL: 0.1483
Entry: 0.15, Exit: 0.4, win rate: 34.29%, PNL: 0.0857
Entry: 0.15, Exit: 0.5, win rate: 30.00%, PNL: 0.1050
Entry: 0.15, Exit: 0.6, win rate: 28.57%, PNL: 0.1286
Entry: 0.15, Exit: 0.7, win rate: 24.29%, PNL: 0.1336
Entry: 0.15, Exit: 0.9, win rate: 21.43%, PNL: 0.1607
Entry: 0.15, Exit: 0.99, win rate: 20.00%, PNL: 0.1680
Entry: 0.2, Exit: 0.4, win rate: 

In [39]:
feature_cols = [  
    'ask_delta',
    'bid_delta',
    'ask_std',
    'bid_std',
    research_side + '_dist',         
    'log_return',  
    'm3_log_return',
    'm5_log_return',  
    'ma3_vs_strike',
    'ma5_vs_strike',            
    research_side + '_dist_pct',
    'm1_' + research_side + '_dist_momentum',
    'm3_' + research_side + '_dist_momentum',
    'm5_' + research_side + '_dist_momentum',
    'time_decay',
    'hour',
    'minute',
    research_side + '_spread',
    'volume_surge',
    'rsi',
    'macd',
    'signal_line',
]

comb_list = [0.05, 0.1, 0.15, 0.2, 0.25,0.3,0.35]
reached_list = ['reached_30','reached_40', 'reached_50', 'reached_60', 'reached_70', 'reached_90', 'reached_99']

for comb in comb_list: 
    for reached in reached_list:
        df_results = pd.DataFrame(res[comb])
        df_results = df_results[df_results['hour'].isin(good_hours)]
        df_expected = df_results[reached]
        
        # Separate good and bad outcomes
        good = df_results[df_expected == 1]
        bad = df_results[df_expected == 0]
        
        # Prior probabilities
        prior_good = len(good) / len(df_results)
        prior_bad = len(bad) / len(df_results)
        print(f"Comb is {comb}, reached is {reached}")
        print(f"Prior P(Good) = {prior_good:.4f}, P(Bad) = {prior_bad:.4f}")
        likelihoods = {}
        good_ratio_count = 0
        for col in feature_cols:
            good_mean, good_std = good[col].mean(), good[col].std()
            bad_mean, bad_std = bad[col].mean(), bad[col].std()
            
            likelihoods[col] = {
                'good': (good_mean, good_std),
                'bad': (bad_mean, bad_std),
            }
            
            diff = good_mean - bad_mean
            ratio, grade = evaluate_feature(good_mean, good_std, bad_mean, bad_std)
            if grade == 'strong' or grade == 'medium':
                good_ratio_count += 1
            print(f"{grade} -> {col}: Good={good_mean:.4f}, Bad={bad_mean:.4f}, ratio={ratio:.4f}")
        print(f"Total Good Ratio is {good_ratio_count}\n")

Comb is 0.05, reached is reached_30
Prior P(Good) = 0.1404, P(Bad) = 0.8596
medium -> ask_delta: Good=-0.0803, Bad=-0.1191, ratio=0.4979
medium -> bid_delta: Good=-0.0734, Bad=-0.1190, ratio=0.4242
weak -> ask_std: Good=0.0803, Bad=0.0968, ratio=0.2426
weak -> bid_std: Good=0.0834, Bad=0.0988, ratio=0.1676
useless -> yes_dist: Good=-79.7575, Bad=-87.2198, ratio=0.1215
useless -> log_return: Good=-0.0003, Bad=-0.0003, ratio=0.0127
medium -> m3_log_return: Good=0.0003, Bad=0.0004, ratio=0.4022
useless -> m5_log_return: Good=0.0005, Bad=0.0005, ratio=0.0034
weak -> ma3_vs_strike: Good=-0.0781, Bad=-0.0967, ratio=0.2594
weak -> ma5_vs_strike: Good=-0.0692, Bad=-0.0862, ratio=0.2345
useless -> yes_dist_pct: Good=-0.1038, Bad=-0.1134, ratio=0.1210
useless -> m1_yes_dist_momentum: Good=-20.3338, Bad=-20.8720, ratio=0.0128
useless -> m3_yes_dist_momentum: Good=-41.2500, Bad=-32.0820, ratio=0.1420
useless -> m5_yes_dist_momentum: Good=-56.6462, Bad=-49.5865, ratio=0.0866
medium -> time_decay: G

In [44]:
df_name = 0.10
df_result_name = 'reached_99'

df_results = pd.DataFrame(res[df_name])
df_results = df_results[df_results['hour'].isin(good_hours)]
df_expected = df_results[df_result_name]

# Separate good and bad outcomes
good = df_results[df_expected == 1]
bad = df_results[df_expected == 0]

# Prior probabilities
prior_good = len(good) / len(df_results)
prior_bad = len(bad) / len(df_results)
likelihoods = {}
params = {}

for col in feature_cols:
    good_mean, good_std = good[col].mean(), good[col].std()
    bad_mean, bad_std = bad[col].mean(), bad[col].std()
    
    likelihoods[col] = {
        'good': (good_mean, good_std),
        'bad': (bad_mean, bad_std),
    }
    
    # diff = good_mean - bad_mean
    ratio, grade = evaluate_feature(good_mean, good_std, bad_mean, bad_std)
    if grade == 'strong' or grade == 'medium':
        print(f"{grade} -> {col}: Good={good_mean:.4f}, Bad={bad_mean:.4f}, Ratio={ratio:.4f}")
        params[col] = {'good': (good_mean, good_std), 'bad': (bad_mean, bad_std),}
# for col in feature_cols:
#     g_mean, g_std = likelihoods[col]['good']
#     b_mean, b_std = likelihoods[col]['bad']
#     params[col] = {
#         'good': (g_mean, g_std),
#         'bad': (b_mean, b_std)
#     }
    
params['period'] = {
    'good': (prior_good,),
    'bad': (prior_bad,)
}
print(f"good is: {prior_good}, bad is: {prior_bad}")

medium -> ma3_vs_strike: Good=-0.0550, Bad=-0.0780, Ratio=0.3124
medium -> ma5_vs_strike: Good=-0.0384, Bad=-0.0628, Ratio=0.4076
medium -> time_decay: Good=6.7273, Bad=5.1636, Ratio=0.4969
strong -> minute: Good=24.6364, Bad=34.1091, Ratio=0.6437
strong -> yes_spread: Good=0.0107, Bad=0.0056, Ratio=0.9992
good is: 0.16666666666666666, bad is: 0.8333333333333334


In [45]:
# type 1 and 2 errors

type1 = 0 
type2 = 0  

total_good = 0
total_bad = 0

feature_names = [name for name in params.keys() if name != 'period']
def predict(*args):
    # Start from prior odds
    log_odds = np.log(params['period']['good'][0] / params['period']['bad'][0])
    
    for name, x in zip(feature_names, args):
        g_m, g_s = params[name]['good']
        b_m, b_s = params[name]['bad']
        
        # Likelihood ratio
        p_g = norm.pdf(x, g_m, g_s + 1e-10)
        p_b = norm.pdf(x, b_m, b_s + 1e-10)
        
        if p_b > 0:
            log_odds += np.log(p_g / p_b)
    
    prob = 1 / (1 + np.exp(-log_odds))
    return prob

for threshold in [0.1, 0.15, 0.2, 0.25, 0.30, 0.35, 0.40, 0.45]:
    type1, type2 = 0, 0
    total_good, total_bad = 0, 0
    
    for index, row in df_results.iterrows(): 
        p = predict(*(row[x] for x in params if x != 'period')) 
        
        actual = row[df_result_name]
        
        if actual == 1:
            total_good += 1
            if p < threshold:
                type1 += 1
        if actual == 0:
            total_bad += 1
            if p >= threshold:
                type2 += 1
    
    bought_good = total_good - type1
    bought_bad = type2
    
    type1_rate = type1 / total_good
    type2_rate = type2 / total_bad
    
    # Precision & Recall
    precision = bought_good / (bought_good + bought_bad) if (bought_good + bought_bad) > 0 else 0
    recall = 1 - type1_rate
    
    # F1
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"Threshold={threshold:.2f}: Type1={type1_rate:.2f}, Type2={type2_rate:.2f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")

Threshold=0.10: Type1=0.09, Type2=0.35, F1=0.5000, Precision=0.3448, Recall=0.9091
Threshold=0.15: Type1=0.09, Type2=0.25, F1=0.5714, Precision=0.4167, Recall=0.9091
Threshold=0.20: Type1=0.27, Type2=0.18, F1=0.5517, Precision=0.4444, Recall=0.7273
Threshold=0.25: Type1=0.45, Type2=0.15, F1=0.4800, Precision=0.4286, Recall=0.5455
Threshold=0.30: Type1=0.45, Type2=0.13, F1=0.5000, Precision=0.4615, Recall=0.5455
Threshold=0.35: Type1=0.55, Type2=0.13, F1=0.4348, Precision=0.4167, Recall=0.4545
Threshold=0.40: Type1=0.55, Type2=0.11, F1=0.4545, Precision=0.4545, Recall=0.4545
Threshold=0.45: Type1=0.64, Type2=0.09, F1=0.4000, Precision=0.4444, Recall=0.3636


In [42]:
# export model

def write_model_to_json(parameters: dict, filepath: str = None):
    if filepath is None:
        filepath = MODEL_PATH / 'yes_bayesian.json'
    
    params_serializable = {}
    for feature, values in parameters.items():
        params_serializable[feature] = {
            'good': list(values['good']),
            'bad': list(values['bad'])
        }
    
    with open(filepath, 'w') as f:
        json.dump(params_serializable, f, indent=2)
    
    print(f"Model saved to {filepath}")


def load_model_from_json(filepath: str = None):
    if filepath is None:
        filepath = MODEL_PATH / 'yes_bayesian.json'
    
    with open(filepath, 'r') as f:
        params_serializable = json.load(f)
    
    # Convert lists back to tuples
    parameters = {}
    for feature, values in params_serializable.items():
        parameters[feature] = {
            'good': tuple(values['good']),
            'bad': tuple(values['bad'])
        }
    
    return parameters


# Correct dict syntax
# params = {
#     'period': {
#         'good': (0.4138,),
#         'bad': (0.5862,)
#     },
#     'm1_yes_dist_momentum': {
#         'good': (-17.5070, 51.8947),
#         'bad':  (-19.7738, 48.4898)
#     },
#     'm3_yes_dist_momentum': {
#         'good': (-29.0380, 82.5982),
#         'bad':  (-32.8337, 73.5574)
#     },
#     'm5_yes_dist_momentum': {
#         'good': (-31.1500, 93.6206),
#         'bad':  (-37.5775, 85.6289)
#     },
#     'yes_dist': {
#         'good': (-19.9004, 29.5222),
#         'bad':  (-22.8592, 38.3313)
#     },
# }

# Write
write_model_to_json(params)
print(params)

Model saved to /Users/yingxie/Documents/Git/Quant/Kalshi/btc_15_strategy/models/yes_bayesian.json
{'ma3_vs_strike': {'good': (-0.05502554991346866, 0.06534699062030032), 'bad': (-0.07801920120532012, 0.08186435722357918)}, 'ma5_vs_strike': {'good': (-0.038416123009839785, 0.052047553036145194), 'bad': (-0.06277846438433961, 0.06749388402790055)}, 'time_decay': {'good': (6.7272727272727275, 3.2891004572955533), 'bad': (5.163636363636364, 3.004710107195506)}, 'minute': {'good': (24.636363636363637, 12.745765785332221), 'bad': (34.10909090909091, 16.686291476308426)}, 'yes_spread': {'good': (0.010727272727272731, 0.005968097001405209), 'bad': (0.005618181818181821, 0.0042578507406076304)}, 'period': {'good': (0.16666666666666666,), 'bad': (0.8333333333333334,)}}


In [32]:
# hour research
reached_list = [50,60,70,90,99]
time_dict = {str(x): 0 for x in range(24)}
test_df = comb_10
test_entry_price = 10

for hour in range(24):
    for reached in reached_list:
        df_h = test_df[test_df['hour'] == hour]
        # if len(df_h) < 2:
        #     continue
        rate = df_h['reached_' + str(reached)].mean()
        ev = rate * ((reached - test_entry_price) / 100)  - (1-rate) * test_entry_price / 100
        time_dict[str(hour)] += ev

sorted_items = sorted(time_dict.items(), key=lambda x: int(x[0]), reverse=True)
for key, item in sorted_items:
    print(f"Hour: {key}, EV: {item:.3f}")

Hour: 23, EV: nan
Hour: 22, EV: nan
Hour: 21, EV: nan
Hour: 20, EV: 1.345
Hour: 19, EV: -0.500
Hour: 18, EV: -0.500
Hour: 17, EV: -0.300
Hour: 16, EV: 0.186
Hour: 15, EV: 0.175
Hour: 14, EV: -0.225
Hour: 13, EV: -0.500
Hour: 12, EV: 1.609
Hour: 11, EV: 0.024
Hour: 10, EV: 0.238
Hour: 9, EV: 0.027
Hour: 8, EV: -0.400
Hour: 7, EV: 0.607
Hour: 6, EV: -0.165
Hour: 5, EV: 0.024
Hour: 4, EV: -0.026
Hour: 3, EV: -0.269
Hour: 2, EV: 0.145
Hour: 1, EV: nan
Hour: 0, EV: nan


In [33]:
# start research
trade_hour = [x for x in range(23)]
reached_dict = {str(x): 0 for x in reached_list}
for hour in trade_hour:
    for reached in reached_list:
        df_h = test_df[test_df['hour'] == hour]
        if len(df_h) < 5:
            continue
        rate = df_h['reached_' + str(reached)].mean()
        ev = rate * ((reached - test_entry_price)/100 - 0.1)  - (1-rate) * test_entry_price / 100
        reached_dict[str(reached)] += ev

for key, item in reached_dict.items():
    print(f"Reached: {key}, EV: {item:.3f}")

Reached: 50, EV: -0.240
Reached: 60, EV: -0.171
Reached: 70, EV: -0.018
Reached: 90, EV: 0.151
Reached: 99, EV: 0.325


In [37]:
# final hour research
trade_hour = [x for x in range(23)]
for hour in trade_hour:
    df_h = test_df[test_df['hour'] == hour]
    # if len(df_h) < 2:
    #     continue
    rate = df_h['reached_99'].mean()
    ev = rate * (0.99 - 0.1)  - (1-rate) * 0.10
    print(f"{hour:02d}:00 {len(df_h):3d} trades win {rate:.0%} | EV {ev:.2f}")

00:00   0 trades win nan% | EV nan
01:00   0 trades win nan% | EV nan
02:00  15 trades win 13% | EV 0.03
03:00  16 trades win 6% | EV -0.04
04:00  16 trades win 6% | EV -0.04
05:00  16 trades win 12% | EV 0.02
06:00  11 trades win 9% | EV -0.01
07:00  10 trades win 30% | EV 0.20
08:00  11 trades win 0% | EV -0.10
09:00  14 trades win 14% | EV 0.04
10:00  10 trades win 20% | EV 0.10
11:00   8 trades win 12% | EV 0.02
12:00   7 trades win 57% | EV 0.47
13:00   9 trades win 0% | EV -0.10
14:00   4 trades win 0% | EV -0.10
15:00   4 trades win 0% | EV -0.10
16:00   8 trades win 12% | EV 0.02
17:00   9 trades win 0% | EV -0.10
18:00   1 trades win 0% | EV -0.10
19:00   3 trades win 0% | EV -0.10
20:00   2 trades win 50% | EV 0.40
21:00   0 trades win nan% | EV nan
22:00   0 trades win nan% | EV nan
